In [ ]:
!pip install -q langgraph langchain langchain-google-genai

In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

In [ ]:
from typing import TypedDict

class AgentState(TypedDict):
    problem: str
    algorithm: str
    plan: str
    code: str
    review: str
    retries: int
    complexity: str
    observations: list[str]

In [ ]:
def analyze_problem(state):

    prompt = f"""
    Analyze this Codeforces problem:

    {state['problem']}

    Find:
    - constraints
    - algorithm
    """

    response = llm.invoke(prompt)

    return {
        **state,
        "algorithm": response.content
    }

In [ ]:
def planner(state):

    prompt = f"""
    Problem:
    {state['problem']}

    Analysis:
    {state['algorithm']}

    Create a detailed competitive programming solution plan.
    """

    response = llm.invoke(prompt)

    return {
        **state,
        "plan": response.content
    }

In [ ]:
def generate_code(state):

    prompt = f"""
    Problem:
    {state['problem']}

    Solution Plan:
    {state['plan']}

    Write a complete C++17 solution.
    Output only code.
    """

    response = llm.invoke(prompt)

    return {
        **state,
        "code": response.content
    }

In [ ]:
def review_code(state):

    prompt = f"""
    You are a Codeforces expert.

    Problem:
    {state['problem']}

    Solution:
    {state['code']}

    Check:
    1. Logical correctness
    2. Edge cases
    3. Time complexity

    Reply exactly in this format:

    STATUS: PASS

    or

    STATUS: FAIL

    Then explain.
    """

    response = llm.invoke(prompt)

    return {
        **state,
        "review": response.content
    }

In [ ]:
def fix_code(state):

    prompt = f"""
    Problem:
    {state['problem']}

    Current Code:
    {state['code']}

    Review:
    {state['review']}

    Fix the solution.

    Return only corrected C++ code.
    """

    response = llm.invoke(prompt)

    return {
        **state,
        "code": response.content,
        "retries": state["retries"] + 1
    }

In [ ]:
def route_after_review(state):

    if "STATUS: PASS" in state["review"]:
        return "end"

    if state["retries"] >= 3:
        return "end"

    return "fix"

In [ ]:
from langgraph.graph import StateGraph, END

builder = StateGraph(AgentState)

builder.add_node("analyze", analyze_problem)
builder.add_node("plan", planner)
builder.add_node("generate", generate_code)
builder.add_node("review", review_code)
builder.add_node("fix", fix_code)

builder.set_entry_point("analyze")

builder.add_edge("analyze", "plan")
builder.add_edge("plan", "generate")
builder.add_edge("generate", "review")
builder.add_conditional_edges(
    "review",
    route_after_review,
    {
        "fix": "fix",
        "end": END
    }
)

builder.add_edge("fix", "review")

graph = builder.compile()

In [ ]:
problem = """
You are given an array of n integers.

Find the maximum element in the array.

Input:
The first line contains an integer n.
The second line contains n space-separated integers.

Output:
Print the maximum element.

Constraints:
1 <= n <= 100000
-10^9 <= a[i] <= 10^9

Example Input:
5
1 8 3 2 7

Example Output:
8
"""

In [ ]:
result = graph.invoke({
    "problem": problem,
    "algorithm": "",
    "plan": "",
    "code": "",
    "review": "",
    "retries": 0
})

print("ALGORITHM:")
print(result["algorithm"])

print("\nPLAN:")
print(result["plan"])

print("\nCODE:")
print(result["code"])

print("\nREVIEW:")
print(result["review"])

print("\nRETRIES:")
print(result["retries"])

ALGORITHM:
## Analysis of "Find the maximum element in the array"

This is a fundamental problem in computer science, often used as an introductory exercise for array manipulation and basic algorithms.

---

### Constraints:

1.  **`1 <= n <= 100000`**:
    *   `n` represents the number of elements in the array.
    *   The array will always have at least one element (`n >= 1`), so we don't need to handle an empty array case.
    *   The maximum size of the array is 100,000 elements. This implies that an algorithm with linear time complexity (O(n)) will be efficient enough. An algorithm with quadratic complexity (O(n^2)) would be too slow (10^5 * 10^5 = 10^10 operations).

2.  **`-10^9 <= a[i] <= 10^9`**:
    *   `a[i]` represents the value of an element in the array.
    *   Elements can be negative, zero, or positive.
    *   The range of values is quite large, requiring a data type that can hold numbers up to `10^9`. In C++ and Java, a standard `int` type is usually sufficient (typi